# Polymer Markov-Corrected MPC Unified Prototype

Prediction-error-validated Markov-parameter correction for polymer offset-free MPC. This notebook is intentionally polymer-only and delegates reproducible implementation details to `report/scripts/generate_polymer_markov_correction_assets.py` so shared RL/MPC modules remain untouched.

## 1. Imports And Configuration

Defaults are loaded from the current polymer baseline configuration. The default run is the existing disturbed profile with `n_tests=200`, `set_points_len=400`, `predict_h=9`, and `cont_h=3`.

In [ ]:
from pathlib import Path

from report.scripts.generate_polymer_markov_correction_assets import (
    build_config,
    run_polymer_markov_correction,
)

CONFIG = build_config()
CONFIG

In [ ]:
# Optional smoke-test override. Leave empty for the full current polymer default run.
# Example for a quick development check:
# CONFIG_OVERRIDES = {"max_steps": 40, "make_plots": True, "save_outputs": True}
CONFIG_OVERRIDES = {}

effective_config = build_config(CONFIG_OVERRIDES)
print("run_mode:", effective_config["run_mode"])
print("n_tests:", effective_config["n_tests"])
print("set_points_len:", effective_config["set_points_len"])
print("predict_h:", effective_config["predict_h"])
print("cont_h:", effective_config["cont_h"])
print("basis_family:", effective_config["basis_family"])
print("max_steps:", effective_config["max_steps"])

## 2. Load Polymer Model And Baseline Objects

The runner loads `POLYMER_BASELINE_DEFAULTS`, `PolymerCSTR`, scaling artifacts, augmented offset-free matrices, input bounds, setpoint schedules, and the canonical baseline path through the existing polymer helper layer.

## 3. Local Markov Helper Functions

The helper functions are isolated in the new report script, not in shared source modules. They implement Markov blocks, lifted Toeplitz maps for absolute input-deviation levels, prediction scoring, gain drift, constrained LS, and lifted MPC cost using the same convention as `MpcSolverGeneral.mpc_opt_fun`.

## 4-8. Run TD3-Assisted Markov MPC

The runner executes the RL-assisted Markov correction loop: lifted-prediction validation, shadow scoring, constrained LS teacher fitting, TD3 Markov action selection, safety-filtered corrected MPC, replay updates, result saving, and standard MPC comparison plotting. TD3 Markov proposals are enabled by default, with constrained LS retained as the warm-start teacher and safety fallback.

In [ ]:
RUN_OUTPUT = run_polymer_markov_correction(CONFIG_OVERRIDES)
RUN_OUTPUT["episode_reward_table"]

## 9. Saved Outputs

The run saves its result bundle, Markov diagnostic figures, and CSV summaries under `Polymer/Results/polymer_markov_corrected_mpc/YYYYMMDD_HHMMSS/`. Standard MPC comparison plots are saved under `Polymer/Results/polymer_markov_compare_<run_mode>/YYYYMMDD_HHMMSS/`, and the living report remains at `report/polymer_markov_correction_progress.md`.

In [ ]:
print("Report:", RUN_OUTPUT["report_path"])
print("Result folder:", RUN_OUTPUT["result_dir"])
print("Result bundle:", RUN_OUTPUT["result_path"])
print("Comparison folder:", RUN_OUTPUT["comparison_dir"])
print("Figures:")
for path in RUN_OUTPUT["figures"]:
    print(" -", path)